# TOBi Routing & Escalation — Executive Management Dashboard

A BigQuery-connected dashboard that answers the five business objectives. Every
panel runs a small **aggregate query** (it never loads the full session table),
and each query is preceded by a plain-English **why / what / how-to-read** note.

| Obj | Question | Section |
|---|---|---|
| 1 | Where are technical issues incorrectly routed/escalated? | 4, 5 |
| 2 | What does misrouting cost (time / channel load / repeats)? | 6, 10 |
| 3 | What drives the wrong routing (topic, entry point, segment)? | 5, 7, 8 |
| 4 | What should we change (routing logic / escalation)? | 9 |
| 5 | How do we track first-contact resolution & efficiency? | 11 |

**How the notebook flows:** Step 1 connect → Step 2 configure → Step 3 build the
`session_master` table once → Steps 4-11 the charts.

## Step 1 — Install libraries & connect to BigQuery

**Why:** we need the BigQuery client to run queries, pandas to hold the small
result tables, and matplotlib to draw the charts.

In [ ]:
# Run once if the libraries are missing:
# %pip install google-cloud-bigquery db-dtypes pandas matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from google.cloud import bigquery

# brand palette + clean styling
RED='#E60000'; DARK='#25282A'; GREY='#7E8083'; LGREY='#E9EAEC'
GREEN='#009900'; AMBER='#FBA600'; BLUE='#0077C8'; INK='#4A4D4E'
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.edgecolor':LGREY,
  'axes.grid':True,'grid.color':LGREY,'axes.axisbelow':True,'axes.spines.top':False,
  'axes.spines.right':False,'axes.titlesize':13,'axes.titleweight':'bold','figure.dpi':110})

def vlabels(ax,bars,fmt='{:.0f}',horiz=False,pad=3):
    """Print value labels on bars."""
    for b in bars:
        v=b.get_width() if horiz else b.get_height()
        xy=(v,b.get_y()+b.get_height()/2) if horiz else (b.get_x()+b.get_width()/2,v)
        off=(4,0) if horiz else (0,pad)
        ax.annotate(fmt.format(v),xy,xytext=off,textcoords='offset points',
                    va='center',ha=('left' if horiz else 'center'),fontweight='bold',fontsize=9)
print('libraries ready')

## Step 2 — Configure

**Why:** tell the notebook your GCP project, the dataset you can write to (where
the dashboard table will live), and the analysis date window.

- `DATASET` must be a dataset where **you can create tables**.
- `DATE_FROM/DATE_TO` exclude the `null` / `1900-01-01` junk dates seen in the raw data.

In [ ]:
PROJECT = 'vf-pt-copsvertex-live'
DATASET = 'your_dataset'           # <-- CHANGE: a dataset you can write to
TABLE   = 'session_master'
DATE_FROM, DATE_TO = '2024-01-01', '2025-12-31'   # <-- CHANGE to your window

client = bigquery.Client(project=PROJECT)
M = f'`{PROJECT}.{DATASET}.{TABLE}`'                 # fully-qualified table name
W = f"DATE(START_MOMENT) BETWEEN '{DATE_FROM}' AND '{DATE_TO}'"  # reusable date filter
def q(sql):
    """Run a query and return a small DataFrame."""
    return client.query(sql).to_dataframe()
print('will read/write:', M)

## Step 3 — Build the `session_master` table (run ONCE)

**What 'materialise' means:** `standalone/session_master_query.sql` is a big query
that rebuilds every session from the two raw tables and works out the tag / topic /
misroute flags. We run it **once** and save the result as a table, so the charts
below read from a fast table instead of recomputing the pipeline every time.

**Why:** without this, each chart would re-run the heavy pipeline (slow & costly).
Set `BUILD_TABLE=False` after the first successful run to skip rebuilding.

In [ ]:
BUILD_TABLE = True   # set to False once the table exists

import os
if BUILD_TABLE:
    # locate the pipeline SQL whether the notebook runs from repo root or /notebooks
    cands = ['../standalone/session_master_query.sql', 'standalone/session_master_query.sql']
    path = next((p for p in cands if os.path.exists(p)), None)
    assert path, 'session_master_query.sql not found - check the path'
    body = open(path).read().strip().rstrip(';')          # drop trailing ; for CREATE TABLE
    print('building table from', path, '...')
    client.query(f'CREATE OR REPLACE TABLE {M} AS\n{body}').result()
    print('done — session_master table is ready')
else:
    print('skipping build (using existing table)')

In [ ]:
# quick sanity check: row count + a peek
print(q(f'SELECT COUNT(*) AS rows FROM {M}'))
q(f'SELECT SESSION_ID, technical_topic_type, outcome_group, routed_support_type, is_hard_misroute FROM {M} WHERE {W} LIMIT 5')

## Step 4 — Headline KPIs  *(all objectives)*

**Why:** give leadership the size of the problem in one glance.
**What:** how many technical sessions, the share misrouted, the share the bot
resolves itself (FCR), and the extra handovers misrouting creates.
**How to read:** the **red** card is the core problem metric; **green** is the win we want to grow.

In [ ]:
k = q(f'''
SELECT COUNTIF(is_technical_topic) tech,
       COUNTIF(is_technical_topic AND is_bot_contained) contained,
       COUNTIF(is_hard_misroute) hard, COUNTIF(is_soft_misroute) soft,
       SUM(IF(is_hard_misroute OR is_soft_misroute, n_transfers, 0)) extra_handovers
FROM {M} WHERE {W}''').iloc[0]
tech=int(k.tech)
cards=[('Technical sessions',f'{tech/1e6:.1f}M','in window',INK),
       ('Misrouted',f'{100*(k.hard+k.soft)/tech:.0f}%',f'hard {100*k.hard/tech:.0f}% + soft {100*k.soft/tech:.0f}%',RED),
       ('Bot-contained (FCR)',f'{100*k.contained/tech:.0f}%','resolved first contact',GREEN),
       ('Extra handovers',f'{k.extra_handovers/1e3:.0f}k','caused by misroutes',BLUE)]
fig,ax=plt.subplots(1,4,figsize=(15,2.5))
for a,(t,v,s,c) in zip(ax,cards):
    a.axis('off')
    a.add_patch(FancyBboxPatch((.04,.08),.92,.84,boxstyle='round,pad=0.02,rounding_size=0.05',lw=0,fc=LGREY,transform=a.transAxes))
    a.add_patch(FancyBboxPatch((.04,.08),.03,.84,boxstyle='square,pad=0',lw=0,fc=c,transform=a.transAxes))
    a.text(.13,.62,v,fontsize=22,fontweight='bold',color=c,transform=a.transAxes,va='center')
    a.text(.13,.30,t,fontsize=10,fontweight='bold',color=DARK,transform=a.transAxes,va='center')
    a.text(.13,.16,s,fontsize=8.5,color=GREY,transform=a.transAxes,va='center')
plt.tight_layout(); plt.show()

## Step 5 — Misroute rate by technical topic  *(Objectives 1 & 3)*

**Why:** to target fixes at the topics that leak most.
**What:** the % of each technical topic that is hard-misrouted (sent to the wrong human skill).
**How to read:** red bars (>=15%) are priorities — typically the *vague* intents
(`general_fault`, `general_difficulty`) where the bot can't tell what's wrong.

In [ ]:
g = q(f'''
SELECT technical_topic_type, COUNT(*) sessions, COUNTIF(is_hard_misroute) hard,
       ROUND(COUNTIF(is_hard_misroute)/COUNT(*)*100,2) pct
FROM {M} WHERE {W} AND is_technical_topic GROUP BY 1 ORDER BY hard DESC''').sort_values('pct')
fig,ax=plt.subplots(figsize=(11,4))
cols=[RED if p>=15 else (AMBER if p>=8 else BLUE) for p in g.pct]
b=ax.barh(g.technical_topic_type,g.pct,color=cols)
for bar,n in zip(b,g.hard):
    ax.annotate(f'{bar.get_width():.1f}%  ({int(n):,})',(bar.get_width(),bar.get_y()+bar.get_height()/2),
                xytext=(4,0),textcoords='offset points',va='center',fontweight='bold',fontsize=9)
ax.set_title('Hard-misroute rate by technical topic'); ax.set_xlabel('%'); ax.margins(x=0.25)
plt.tight_layout(); plt.show(); g

## Step 6 — The technical-request funnel  *(Objectives 1 & 5)*

**Why:** show the end-to-end fate of a technical request on one bar stack.
**What:** counts at each stage — correctly handled, bot-contained, soft- and hard-misrouted.
**How to read:** the gap between *Technical requests* and *Correctly handled* is the opportunity.

In [ ]:
f = q(f'''
SELECT COUNTIF(is_technical_topic) tech, COUNTIF(is_correct_technical_route) correct,
       COUNTIF(is_technical_topic AND is_bot_contained) contained,
       COUNTIF(is_soft_misroute) soft, COUNTIF(is_hard_misroute) hard
FROM {M} WHERE {W}''').iloc[0]
stages=[('Technical requests',f.tech,INK),('Correctly handled',f.correct,GREEN),
        ('Bot-contained (FCR)',f.contained,BLUE),('Soft misroute',f.soft,AMBER),('Hard misroute',f.hard,RED)]
fig,ax=plt.subplots(figsize=(11,3.8))
for i,(lab,val,c) in enumerate(stages):
    ax.barh(i,val,color=c,height=.62)
    ax.annotate(f'{int(val):,} ({100*val/f.tech:.0f}%)',(val,i),xytext=(6,0),
                textcoords='offset points',va='center',fontweight='bold',fontsize=9)
ax.set_yticks(range(len(stages))); ax.set_yticklabels([s[0] for s in stages]); ax.invert_yaxis()
ax.set_title('What happens to technical requests'); ax.margins(x=0.2); plt.tight_layout(); plt.show()

## Step 7 — Impact: channel load & repeats  *(Objective 2)*

**Why:** quantify the operational cost of misrouting.
**What:** average transfers per session and the 24h repeat-contact rate, by cohort.
**How to read:** misrouted cohorts should show **more transfers and repeats** than
correctly-routed — that's the rework cost (session *duration* is similar, so it's not the lever).

In [ ]:
c = q(f'''
SELECT CASE WHEN is_hard_misroute THEN 'misrouted_hard' WHEN is_soft_misroute THEN 'misrouted_soft'
            WHEN is_correct_technical_route THEN 'correct_technical'
            WHEN is_technical_topic THEN 'technical_other' ELSE 'non_technical' END cohort,
       COUNT(*) sessions, ROUND(AVG(n_transfers),2) avg_transfers,
       ROUND(AVG(CAST(repeat_contact_24h AS INT64))*100,1) pct_repeat
FROM {M} WHERE {W} GROUP BY 1''')
order=['non_technical','technical_other','correct_technical','misrouted_soft','misrouted_hard']
c=c.set_index('cohort').reindex(order).dropna(how='all'); pal=[GREY,BLUE,GREEN,AMBER,RED][:len(c)]
fig,ax=plt.subplots(1,2,figsize=(14,4.2))
b1=ax[0].bar(c.index,c.avg_transfers,color=pal); vlabels(ax[0],b1,'{:.2f}')
ax[0].set_title('Avg transfers / session'); ax[0].tick_params(axis='x',rotation=25,labelsize=8)
b2=ax[1].bar(c.index,c.pct_repeat,color=pal); vlabels(ax[1],b2,'{:.0f}%')
ax[1].set_title('Repeat contact within 24h (%)'); ax[1].tick_params(axis='x',rotation=25,labelsize=8)
plt.tight_layout(); plt.show(); c

## Step 8 — Entry-point driver: channel  *(Objective 3)*

**Why:** find which channels misroute most so fixes are targeted.
**What:** hard-misroute rate by channel (>=5k technical sessions for stability).
**How to read:** high rate **and** high volume (e.g. `voice`) = top priority.
*(Note: `CONFIDENCE_LEVEL` is ~all 0/blank in this data, so it is not a usable driver.)*

In [ ]:
ch = q(f'''
SELECT CHANNEL, COUNTIF(is_technical_topic) tech, COUNTIF(is_hard_misroute) hard,
       ROUND(COUNTIF(is_hard_misroute)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) pct
FROM {M} WHERE {W} GROUP BY CHANNEL HAVING tech>=5000 ORDER BY pct DESC''')
fig,ax=plt.subplots(figsize=(11,4))
b=ax.bar(ch.CHANNEL,ch.pct,color=INK); vlabels(ax,b,'{:.1f}%')
ax.set_title('Misroute % by channel (>=5k technical sessions)'); ax.tick_params(axis='x',rotation=25)
plt.tight_layout(); plt.show(); ch

## Step 9 — EXTRA: misroute by customer segment  *(Objective 3)*

**Why management cares:** shows whether certain customer types (Postpaid / Prepaid /
Fixed / Business) are served worse — useful for prioritising fixes by value segment.
**What:** technical hard-misroute rate by `routed_client_type`.
**How to read:** a high bar for a high-value segment (e.g. Postpaid/Fixed) is a CX risk.

In [ ]:
seg = q(f'''
SELECT routed_client_type AS segment, COUNTIF(is_technical_topic) tech,
       COUNTIF(is_hard_misroute) hard,
       ROUND(COUNTIF(is_hard_misroute)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) pct
FROM {M} WHERE {W} AND routed_client_type IS NOT NULL
GROUP BY 1 HAVING tech>=2000 ORDER BY pct DESC''')
fig,ax=plt.subplots(figsize=(10,3.8))
b=ax.bar(seg.segment, seg.pct, color=BLUE); vlabels(ax,b,'{:.1f}%')
ax.set_title('Hard-misroute % by customer segment'); ax.tick_params(axis='x',rotation=20)
plt.tight_layout(); plt.show(); seg

## Step 10 — EXTRA: where technical requests end up (outcome mix)  *(Objectives 1 & 2)*

**Why management cares:** one picture of how technical demand splits across
bot-containment, digital deflection, assisted deflection, transfers, abandons and errors.
**What:** share of technical sessions by `outcome_group`.
**How to read:** a large *abandoned*/*error* slice or heavy *transfer* share signals lost
self-service opportunity and avoidable human cost.

In [ ]:
o = q(f'''
SELECT COALESCE(outcome_group,'(no tag)') outcome_group, COUNT(*) sessions
FROM {M} WHERE {W} AND is_technical_topic GROUP BY 1 ORDER BY sessions DESC''')
o['pct']=100*o.sessions/o.sessions.sum()
fig,ax=plt.subplots(figsize=(11,4))
b=ax.barh(o.outcome_group[::-1], o.pct[::-1], color=INK)
for bar in b: ax.annotate(f'{bar.get_width():.1f}%',(bar.get_width(),bar.get_y()+bar.get_height()/2),
                          xytext=(4,0),textcoords='offset points',va='center',fontweight='bold',fontsize=9)
ax.set_title('Technical sessions by outcome'); ax.set_xlabel('% of technical'); ax.margins(x=0.2)
plt.tight_layout(); plt.show(); o

## Step 11 — EXTRA: when misroutes happen (weekday x hour heatmap)  *(Objective 2)*

**Why management cares:** misroutes create live-agent load — knowing *when* they
peak supports staffing and flow decisions.
**What:** count of hard misroutes by day-of-week and hour.
**How to read:** bright cells = peak misroute load windows.

In [ ]:
hm = q(f'''
SELECT EXTRACT(DAYOFWEEK FROM START_MOMENT) dow, EXTRACT(HOUR FROM START_MOMENT) hour,
       COUNTIF(is_hard_misroute) hard
FROM {M} WHERE {W} GROUP BY dow, hour''')
piv = hm.pivot_table(index='dow', columns='hour', values='hard', fill_value=0)
fig,ax=plt.subplots(figsize=(13,3.6))
im=ax.imshow(piv, aspect='auto', cmap='magma')
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][:len(piv.index)])
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_title('Hard misroutes by day-of-week x hour'); ax.set_xlabel('hour')
fig.colorbar(im, ax=ax, shrink=.8, label='misroutes'); plt.tight_layout(); plt.show()

## Step 12 — Recommendations: top misroute leaks  *(Objective 4)*

**Why:** turn the analysis into a ranked action list.
**What:** the biggest topic → wrong-destination leaks, ranked by an impact score
(volume + 2x repeats + handovers).
**How to read:** the top rows are the highest-ROI routing-rule changes to ship first.

In [ ]:
leaks = q(f'''
SELECT technical_topic_type, final_transfer_target, routed_support_type,
       COUNT(*) misrouted, COUNTIF(repeat_contact_24h) repeats, SUM(n_transfers) handovers,
       COUNT(*) + 2*COUNTIF(repeat_contact_24h) + SUM(n_transfers) AS impact_score
FROM {M} WHERE {W} AND is_hard_misroute GROUP BY 1,2,3 ORDER BY impact_score DESC LIMIT 15''')
top=leaks.head(10).iloc[::-1]
fig,ax=plt.subplots(figsize=(12,4.6))
b=ax.barh(top.technical_topic_type+' -> '+top.final_transfer_target, top.misrouted, color=RED)
vlabels(ax,b,'{:,.0f}',horiz=True)
ax.set_title('Top 10 misroute leaks'); ax.set_xlabel('misrouted sessions'); ax.margins(x=0.2)
ax.tick_params(axis='y',labelsize=8); plt.tight_layout(); plt.show(); leaks

## Step 13 — Implementation tracking: FCR vs misroute over time  *(Objective 5)*

**Why:** prove whether shipped fixes work.
**What:** 7-day rolling FCR % and hard-misroute % by day.
**How to read:** after fixes go live, the **red line should fall** and the **green line rise**.

In [ ]:
t = q(f'''
SELECT DATE(START_MOMENT) day,
       ROUND(COUNTIF(is_technical_topic AND is_bot_contained)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) fcr_pct,
       ROUND(COUNTIF(is_hard_misroute)/NULLIF(COUNTIF(is_technical_topic),0)*100,2) misroute_pct
FROM {M} WHERE {W} GROUP BY day ORDER BY day''')
t['day']=pd.to_datetime(t.day)
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(t.day, t.misroute_pct.rolling(7,min_periods=1).mean(), color=RED, lw=2.4, label='Hard-misroute % (7d)')
ax.plot(t.day, t.fcr_pct.rolling(7,min_periods=1).mean(), color=GREEN, lw=2.4, label='FCR % (7d)')
ax.set_title('Technical misroute vs FCR over time'); ax.set_ylabel('%'); ax.legend(frameon=False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

---
*Source: `session_master` (corrected pipeline; see `docs/tag_mappings.md`). Technical
topic from `S_` entity ids; routing from the `T_` tag outcome + support type. Cost
driver = handovers + repeats, not session length.*